In [ ]:
"""Unsupervised Proximity Forest features for time series collections.

Two label-free transformers built on aeon's elastic distances:

  ProximityForestEmbedding   Proximity Forest with the Gini criterion replaced
                             by a balance heuristic. Routes each series to a
                             leaf in every tree; the concatenated one-hot leaf
                             indicator is the feature vector. This is the
                             direct analogue of sklearn's RandomTreesEmbedding.

  ElasticComparisonFeatures  The Proximity Forest split rule on its own:
                             d(x, e1) - d(x, e2) for random exemplar pairs,
                             random measures and random parameters. Dense,
                             flat, no tree machinery.

Neither touches y at any point, so they can be fitted on train + unlabeled
test data when the protocol allows it.

Input is a 3D numpy array (n_cases, n_channels, n_timepoints).
"""

from __future__ import annotations

import numpy as np
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted, check_random_state

from aeon.distances import pairwise_distance

DEFAULT_MEASURES = ("euclidean", "dtw", "ddtw", "wdtw", "lcss", "erp", "msm", "twe")


def sample_measure(rng, sigma, measures=DEFAULT_MEASURES):
    """Draw a measure and its parameters, following the Proximity Forest paper."""
    name = measures[rng.randint(len(measures))]
    if name in ("dtw", "ddtw"):
        return name, {"window": rng.uniform(0.0, 0.25)}
    if name in ("wdtw", "wddtw"):
        return name, {"g": rng.uniform(0.0, 1.0)}
    if name == "lcss":
        return name, {"epsilon": rng.uniform(0.2 * sigma, sigma),
                      "window": rng.uniform(0.0, 0.25)}
    if name == "erp":
        return name, {"g": rng.uniform(0.2 * sigma, sigma)}
    if name == "msm":
        return name, {"c": float(10 ** rng.uniform(-2, 2))}
    if name == "twe":
        return name, {"nu": float(10 ** rng.uniform(-5, -1)),
                      "lmbda": float(rng.randint(10)) / 9.0}
    return name, {}


class _Node:
    __slots__ = ("exemplars", "method", "params", "children", "leaf_id")

    def __init__(self, exemplars=None, method=None, params=None,
                 children=None, leaf_id=None):
        self.exemplars = exemplars
        self.method = method
        self.params = params
        self.children = children
        self.leaf_id = leaf_id

    @property
    def is_leaf(self):
        return self.children is None


def _balance(assign, k):
    """Normalised entropy of a partition. 1.0 is a perfectly even split."""
    counts = np.bincount(assign, minlength=k).astype(float)
    p = counts[counts > 0] / counts.sum()
    if p.size < 2:
        return 0.0
    return float(-(p * np.log(p)).sum() / np.log(k))


class ProximityForestEmbedding(TransformerMixin, BaseEstimator):
    """Unsupervised Proximity Forest leaf-indicator embedding.

    Parameters
    ----------
    n_trees : int
        Number of proximity trees.
    branching : int
        Exemplars per node, i.e. children per internal node. In supervised PF
        this equals the number of classes; here it is a free parameter.
    max_depth : int
        Maximum tree depth. Leaves per tree is at most branching ** max_depth.
    min_samples_split : int
        Nodes with fewer cases become leaves.
    n_candidates : int
        Splitter candidates evaluated per node. The most balanced one wins.
        Set to 1 for fully random splits (cheaper, weaker).
    min_balance : float
        Reject splits whose normalised partition entropy falls below this.
    measures : tuple of str
        Elastic measures to sample from.
    sparse_output : bool
        Return a CSR indicator matrix rather than a dense array.
    """

    def __init__(self, n_trees=100, branching=3, max_depth=5,
                 min_samples_split=5, n_candidates=5, min_balance=0.25,
                 measures=DEFAULT_MEASURES, sparse_output=True,
                 n_jobs=1, random_state=None):
        self.n_trees = n_trees
        self.branching = branching
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_candidates = n_candidates
        self.min_balance = min_balance
        self.measures = measures
        self.sparse_output = sparse_output
        self.n_jobs = n_jobs
        self.random_state = random_state

    # ---------------------------------------------------------------- fit

    def _assign(self, X, exemplars, method, params):
        d = pairwise_distance(X, exemplars, method=method,
                              n_jobs=self.n_jobs, **params)
        return np.argmin(d, axis=1)

    def _grow(self, X, idx, depth, rng, counter):
        if depth >= self.max_depth or len(idx) < self.min_samples_split:
            leaf = _Node(leaf_id=counter[0])
            counter[0] += 1
            return leaf

        best = None
        for _ in range(self.n_candidates):
            k = min(self.branching, len(idx))
            e_idx = rng.choice(idx, size=k, replace=False)
            method, params = sample_measure(rng, self._sigma, self.measures)
            assign = self._assign(X[idx], X[e_idx], method, params)
            score = _balance(assign, k)
            if best is None or score > best[0]:
                best = (score, e_idx, method, params, assign, k)

        score, e_idx, method, params, assign, k = best
        if score < self.min_balance:            # degenerate, stop here
            leaf = _Node(leaf_id=counter[0])
            counter[0] += 1
            return leaf

        children = [
            self._grow(X, idx[assign == b], depth + 1, rng, counter)
            if np.any(assign == b) else
            _Node(leaf_id=self._new_leaf(counter))
            for b in range(k)
        ]
        return _Node(exemplars=X[e_idx], method=method, params=params,
                     children=children)

    @staticmethod
    def _new_leaf(counter):
        counter[0] += 1
        return counter[0] - 1

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        rng = check_random_state(self.random_state)
        self._sigma = float(np.std(X)) or 1.0

        self.trees_, self.n_leaves_ = [], []
        n = len(X)
        for _ in range(self.n_trees):
            counter = [0]
            root = self._grow(X, np.arange(n), 0, rng, counter)
            self.trees_.append(root)
            self.n_leaves_.append(counter[0])
        self.offsets_ = np.cumsum([0] + self.n_leaves_)
        self.n_features_out_ = int(self.offsets_[-1])
        return self

    # ---------------------------------------------------------- transform

    def _route(self, X, node, idx, out):
        if node.is_leaf:
            out[idx] = node.leaf_id
            return
        assign = self._assign(X[idx], node.exemplars, node.method, node.params)
        for b, child in enumerate(node.children):
            sub = idx[assign == b]
            if sub.size:
                self._route(X, child, sub, out)

    def transform(self, X):
        check_is_fitted(self, "trees_")
        X = np.asarray(X, dtype=float)
        n = len(X)
        cols = np.empty((n, self.n_trees), dtype=np.int64)
        for t, root in enumerate(self.trees_):
            leaves = np.empty(n, dtype=np.int64)
            self._route(X, root, np.arange(n), leaves)
            cols[:, t] = leaves + self.offsets_[t]

        rows = np.repeat(np.arange(n), self.n_trees)
        M = sparse.csr_matrix(
            (np.ones(n * self.n_trees, dtype=np.float32),
             (rows, cols.ravel())),
            shape=(n, self.n_features_out_),
        )
        return M if self.sparse_output else M.toarray()

    def proximity(self, X):
        """Pairwise proximity: fraction of trees where two series share a leaf."""
        M = self.transform(X)
        return np.asarray((M @ M.T).todense()) / self.n_trees


class ElasticComparisonFeatures(TransformerMixin, BaseEstimator):
    """Signed exemplar-pair comparisons: d(x, e1) - d(x, e2), no labels used."""

    def __init__(self, n_features=1000, measures=DEFAULT_MEASURES,
                 normalise=True, n_jobs=1, random_state=None):
        self.n_features = n_features
        self.measures = measures
        self.normalise = normalise
        self.n_jobs = n_jobs
        self.random_state = random_state

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        rng = check_random_state(self.random_state)
        self._sigma = float(np.std(X)) or 1.0
        self.pairs_ = []
        for _ in range(self.n_features):
            i, j = rng.choice(len(X), size=2, replace=False)
            method, params = sample_measure(rng, self._sigma, self.measures)
            self.pairs_.append((X[[i]], X[[j]], method, params))
        return self

    def transform(self, X):
        check_is_fitted(self, "pairs_")
        X = np.asarray(X, dtype=float)
        out = np.empty((len(X), len(self.pairs_)), dtype=float)
        for f, (e1, e2, method, params) in enumerate(self.pairs_):
            d1 = pairwise_distance(X, e1, method=method,
                                   n_jobs=self.n_jobs, **params)[:, 0]
            d2 = pairwise_distance(X, e2, method=method,
                                   n_jobs=self.n_jobs, **params)[:, 0]
            out[:, f] = (d1 - d2) / (d1 + d2 + 1e-12) if self.normalise else d1 - d2
        return out